# 8-bob. 8-BOB. ZARARLI DASTURLARNI ANIQLASH II: DINAMIK TAHLIL VA CHUQUR O'QITISH
## Kengaytirilgan amaliy ish: n-gram modeli va LSTM axlat chaqiruvlarga qarshi

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/08-bob/8-2_n-gram-modeli-va-lstm.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 8-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dasturda zararli izlarda jarayonga kod kiritish zanjiri (OpenProcess → VirtualAllocEx → WriteProcessMemory → CreateRemoteThread) to'g'ri tartibda uchraydi, zararsiz izlarda esa xuddi shu to'rtta chaqiruv boshqa tartibda keladi. Demak, chaqiruvlar to'plami bir xil, farq faqat tartibda. Zanjir qadamlari orasiga "axlat" chaqiruvlar qo'shiladi. Ikkita model taqqoslanadi: chaqiruvlar 1–3 gramlari asosidagi logistik regressiya va PyTorch kutubxonasidagi kichik LSTM tarmog'i. Dastur ikki rejimda ishga tushiriladi: birinchisida o'qitish izlarida qadamlar orasida 0 yoki aniq 1 ta axlat chaqiruv bor, ikkinchisida ("xilma-xil") har bir qadamdan keyin 0–2 ta tasodifiy son. Sinovda esa o'qitishda bo'lmagan 4 va 6 ta axlat chaqiruvli izlar ham ishlatiladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
import torch
from torch import nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

rng = np.random.default_rng(8)
torch.manual_seed(8)
# Ishga tushirish: python l.py  yoki  python l.py xilma-xil

In [2]:
# 1. Sun'iy API chaqiruvlari izlari
FON = ["NtReadFile", "NtWriteFile", "RegQueryValue", "NtCreateFile", "NtClose", "LdrLoadDll",
       "GetSystemTime", "NtQueryInformationProcess", "OpenProcess", "VirtualAllocEx",
       "WriteProcessMemory", "CreateRemoteThread", "NtAllocateVirtualMemory", "NtProtectVirtualMemory"]
KIRITISH = ["OpenProcess", "VirtualAllocEx", "WriteProcessMemory", "CreateRemoteThread"]  # jarayonga kod kiritish

def iz(zararli, uzunlik=60, axlat=0):
    s = list(rng.choice(FON, uzunlik))
    if zararli:                      # kod kiritish zanjiri to'g'ri tartibda, orasiga "axlat" chaqiruvlar
        k = rng.integers(0, uzunlik)
        zanjir = []
        for qadam in KIRITISH:
            n_axlat = axlat if axlat >= 0 else rng.integers(0, 3)     # -1: har qadamda 0-2 ta tasodifiy
            zanjir += [qadam] + list(rng.choice(FON[:8], n_axlat))
        s[k:k] = zanjir
    else:                            # zararsiz dastur ham shu chaqiruvlarni ishlatadi, lekin boshqa tartibda
        k = rng.integers(0, uzunlik)
        s[k:k] = list(rng.permutation(KIRITISH))
    return s

def toplam(n, axlat=0):
    X = [iz(i % 2 == 1, axlat=axlat) for i in range(n)]
    return X, np.arange(n) % 2

# o'qitish to'plami: chaqiruvlar orasida 0 yoki 1 ta "axlat" chaqiruv bo'lgan izlar
import sys
XILMA_XIL = len(sys.argv) > 1 and sys.argv[1] == "xilma-xil"
if XILMA_XIL:   # har bir qadamdan keyin 0-2 ta tasodifiy "axlat" chaqiruv
    X_o, y_o = toplam(2000, axlat=-1)
else:           # yarmida axlat yo'q, yarmida har qadamdan keyin aniq 1 ta
    X0, y0 = toplam(1000, axlat=0); X1, y1 = toplam(1000, axlat=1)
    X_o, y_o = X0 + X1, np.r_[y0, y1]
sinovlar = {f"axlat={a}": toplam(1000, axlat=a) for a in (0, 1, 2, 4, 6)}   # 2 va 4 - o'qitishda bo'lmagan

In [3]:
# 2. Model 1: chaqiruvlar 1-3 gramlari + logistik regressiya
vek = CountVectorizer(tokenizer=lambda s: s, preprocessor=None, lowercase=False,
                      token_pattern=None, ngram_range=(1, 3), binary=True)
lr = LogisticRegression(max_iter=3000).fit(vek.fit_transform(X_o), y_o)

In [4]:
# 3. Model 2: LSTM
lugat = {api: i + 1 for i, api in enumerate(FON)}
def tensor(X):
    L = max(map(len, X))
    return torch.tensor([[lugat[a] for a in s] + [0] * (L - len(s)) for s in X])
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(lugat) + 1, 16, padding_idx=0)
        self.lstm = nn.LSTM(16, 32, batch_first=True)
        self.chiq = nn.Linear(32, 1)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.chiq(h.max(1).values).squeeze(1)      # vaqt bo'yicha maksimal qiymat
model = LSTM()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt, yt = tensor(X_o), torch.tensor(y_o, dtype=torch.float32)
for davr in range(15):
    for i in torch.randperm(len(Xt)).split(64):
        opt.zero_grad()
        nn.functional.binary_cross_entropy_with_logits(model(Xt[i]), yt[i]).backward()
        opt.step()

print(f"{'Sinov':<10}{'n-gram + LR':>14}{'LSTM':>8}   (F1)")
for nom, (X, y) in sinovlar.items():
    f_lr = f1_score(y, lr.predict(vek.transform(X)))
    with torch.no_grad():
        f_lstm = f1_score(y, (model(tensor(X)) > 0).int().numpy())
    print(f"{nom:<10}{f_lr:>14.2f}{f_lstm:>8.2f}")

Sinov        n-gram + LR    LSTM   (F1)


axlat=0             0.91    0.95


axlat=1             0.88    0.97


axlat=2             0.60    0.05


axlat=4             0.64    0.05


axlat=6             0.62    0.05


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Sinov        n-gram + LR    LSTM   (F1)
axlat=0             0.91    0.95
axlat=1             0.88    0.97
axlat=2             0.60    0.05
axlat=4             0.64    0.05
axlat=6             0.62    0.05
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 8-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.